# 03 — Zero-shot classification with CLIP and RemoteCLIP

**Question this notebook answers:** can a vision-language model classify
Sentinel-2 land cover with **no training data at all**, and does a
remote-sensing-specific CLIP do better than a generic one?

**Produces:** the zero-shot numbers that anchor notebook 04's headline figure,
a prompt-engineering table, embedding-space projections, and error analysis.
Metrics go to `outputs/results.json`, figures to `figures/03_*.png`.

**Expected runtime:** ~20 minutes on a Colab T4 (mostly image encoding).

### Environment setup and persistence

On Colab this clones the repository, installs the pinned dependencies, and — the
part that matters — mounts Google Drive so that **outputs survive the session**.
Locally it is a no-op beyond putting `src/` on the path.

**Why Drive.** A Colab VM is deleted when the session ends, and the notebooks
depend on each other's artefacts: 01 writes the split and normalisation
statistics, 02 writes the model checkpoint that 04, 05 and 06 all load. Without
persistence, a disconnect means re-running earlier chapters. So `outputs/` and
`figures/` are redirected to Drive via environment variables read by
`s2map.config` at import time.

**`data/` is deliberately NOT on Drive.** The EuroSAT cache is ~2.9 GB and every
training epoch reads it randomly; Drive is a network mount and random reads
through it are slow enough to dominate the runtime. Re-downloading into the
local VM disk each session costs a few minutes and is the faster trade. Set
`USE_DRIVE = False` to keep everything ephemeral.

The install is wrapped so a fragile wheel prints a clear message instead of
killing the kernel halfway through a 40-minute session.

**If this cell reports a numpy mismatch**, restart the runtime
(*Runtime → Restart session*) and run it again. Colab preinstalls a mutually
consistent numpy/torch/scipy set; when pip replaces one of them on disk, the
kernel is left holding the old version in memory and every compiled extension
raises `ValueError: numpy.dtype size changed`. Only a restart fixes it. The
requirements file uses compatible *ranges* rather than exact pins for exactly
this group of packages, so it should not happen — the check is there because it
is the single most common way a Colab notebook dies.

In [ ]:
# --- edit these two if you are running your own fork -----------------------
GITHUB_USER = "SaadH-077"
USE_DRIVE = True          # False -> everything stays in the ephemeral session
# ---------------------------------------------------------------------------

import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO = "s2-chips-to-map"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PERSIST = Path("/content/drive/MyDrive/s2-chips-to-map")
        for sub in ("outputs", "figures"):
            (PERSIST / sub).mkdir(parents=True, exist_ok=True)
        # Read by s2map.config at import time, so this must happen before the import below.
        os.environ["S2MAP_OUTPUT_DIR"] = str(PERSIST / "outputs")
        os.environ["S2MAP_FIGURE_DIR"] = str(PERSIST / "figures")
        print("persisting outputs and figures to", PERSIST)

    if not Path(REPO).exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=False)
    if Path(REPO).exists():
        os.chdir(REPO)
        # Pick up fixes without needing a fresh VM: a stale clone from an earlier
        # session is otherwise invisible and confusing.
        subprocess.run(["git", "pull", "--ff-only", "--quiet"], check=False)
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    except subprocess.CalledProcessError as exc:
        print("!! dependency install failed:", exc)
        print("!! continuing anyway — the cells below will report what is missing")

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from s2map import config as C
C.ensure_dirs()
C.print_environment()
print()
# Catches the one Colab failure mode that no amount of re-running fixes:
# pip replaced numpy on disk after this kernel already imported it.
C.check_environment()
cfg = C.load_config()
SEED = C.set_seed(cfg.seed)
print(f"seed             {SEED}   (multi-seed runs use {cfg.seeds})")
DEVICE = C.get_device()
print(f"device           {DEVICE}")
print(f"outputs ->       {C.OUTPUT_DIR}")
print(f"figures ->       {C.FIGURE_DIR}")
print(f"data cache ->    {C.DATA_DIR}  (local/ephemeral by design — see the note above)")
if DEVICE == "cpu":
    print("\n!! no GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.")

### How zero-shot classification with CLIP actually works

CLIP is trained contrastively on hundreds of millions of image–caption pairs, to
place an image and its own caption close together in a shared embedding space
and everything else far apart. Nothing in that objective is about
classification — but it hands you a classifier for free:

1. Write one text prompt per class: *"a satellite photo of a forest."*
2. Encode each prompt with the text encoder → one vector per class.
3. Encode the image with the image encoder → one vector.
4. Predict the class whose text vector has the highest cosine similarity.

No classifier head, no gradient step, no labels. The "weights" of the classifier
are literally the text embeddings, which means the *words you choose are part of
the model* — a fact section 3 turns into an experiment.

**The prediction for this domain.** CLIP's training distribution is internet
photographs: eye-level, RGB, of things people photograph and caption. Overhead
10 m multispectral imagery is far outside it — there are comparatively few
captioned satellite images on the web, and none of them are 64×64 crops of a
Bavarian field. So expect mediocre absolute performance. That expectation is
exactly the motivation for domain-adapted variants such as RemoteCLIP, which is
the second half of this notebook.

In [ ]:
import numpy as np
from s2map import bands as B, clip_utils as CL, data as D, evaluate as E, viz as V

V.set_style()
bundle = D.load_eurosat_ms()
X, y = bundle.X, bundle.y
splits = D.load_splits()

# Zero-shot needs no training data, so we could evaluate on everything. We use
# the same held-out test split as every other chapter, so the numbers are
# directly comparable with notebooks 02 and 04.
test_idx = splits["test"]
val_idx = splits["val"]
print(f"test chips {test_idx.size:,}   val chips {val_idx.size:,}")

### The preprocessing mismatch — say the cost out loud

| | CLIP expects | EuroSAT provides |
|---|---|---|
| channels | 3 (RGB) | 13 (multispectral) |
| size | 224×224 | 64×64 |
| dtype | 8-bit, CLIP-normalised | 16-bit reflectance |

Bridging the gap: select B04/B03/B02 → 2–98% percentile stretch → quantise to
8-bit → bicubic resize to 224 → CLIP's normalisation.

**Ten of the thirteen bands are discarded.** Every band that notebook 01 showed
carries the separability — the red edge, the NIR that makes the vegetation jump,
the SWIR that tracks moisture and built-up surfaces — is thrown away, because
the vision-language model can only look at something shaped like a photograph.
This is a *structural* limitation of applying RGB-pretrained VLMs to
multispectral data, not a tuning problem: no prompt can recover a band the model
cannot accept.

It also fixes what a fair comparison looks like. Against notebook 02's 13-band
ResNet, CLIP is fighting with one hand tied. The apples-to-apples comparison is
against the **RGB-only** ResNet ablation, and both are reported below.

The stretch is not optional either: feeding raw reflectance would hand CLIP a
near-black image, i.e. put it even further out of distribution than it already
is.

In [ ]:
model, preprocess, tokenizer = CL.load_openclip("ViT-B-32", pretrained="openai", device=DEVICE)
print("preprocess pipeline:", preprocess)

demo = np.asarray(X[int(test_idx[0])])
pil = CL.ms_chip_to_pil(demo)
print(f"chip {demo.shape} {demo.dtype} -> PIL {pil.size} {pil.mode} -> tensor {tuple(preprocess(pil).shape)}")
print(f"bands kept: {B.RGB_BANDS}  bands discarded: "
      f"{tuple(b for b in B.BAND_IDS if b not in B.RGB_BANDS)}")

Encoding is the expensive part, so it is done once per model over the test split
and reused by every prompt strategy below — the four strategies differ only in
the *text* side of the comparison. Chips are processed in chunks of 1,024 so the
224×224 float tensors never all coexist in memory: 4,050 test chips at
3×224×224 float32 would be ~2.4 GB in one go.

In [ ]:
import time

def encode_split(model, preprocess, idx, batch=256, chunk=1024):
    """Encode chips in chunks so the 224x224 float tensors never all coexist."""
    feats = []
    t0 = time.time()
    for s in range(0, idx.size, chunk):
        block = np.asarray(X[idx[s:s + chunk]])
        images = CL.preprocess_chips(block, preprocess, batch=batch)
        feats.append(CL.encode_images(model, images, device=DEVICE, batch=batch))
    import torch
    out = torch.cat(feats)
    print(f"encoded {idx.size:,} chips -> {tuple(out.shape)} in {time.time() - t0:.0f}s")
    return out

clip_test_feats = encode_split(model, preprocess, test_idx)
y_test = y[test_idx]
assert clip_test_feats.shape[0] == y_test.size

## The prompt-engineering study

Four strategies, evaluated under an identical protocol. This is a real
experiment, not a garnish: in a zero-shot system the class name **is** a model
parameter, and the size of the effect surprises people.

- **v1 — raw dataset labels.** `"AnnualCrop"`, `"HerbaceousVegetation"`. These
  are CamelCase directory names, not English. CLIP's tokeniser will shred them
  into subword fragments that never co-occurred in any caption.
- **v2 — simple template**, raw labels: `"a satellite photo of {}."`
- **v3 — natural-language class names** in the same template: `AnnualCrop` →
  *"annual cropland with seasonal crops"*. Comparing v2 with v3 isolates the
  effect of the **wording** with the template held fixed.
- **v4 — prompt ensembling.** Ten templates per class, embeddings averaged and
  L2-renormalised. This is the technique from the original CLIP paper: averaging
  cancels the idiosyncratic direction each phrasing contributes and leaves the
  part that is actually about the class.

The renormalisation after averaging matters — the mean of unit vectors is not a
unit vector, and skipping it would systematically penalise classes whose prompts
disagree with each other, biasing the classifier by prompt consistency rather
than by image content.

In [ ]:
from IPython.display import Markdown, display

prompt_results = {}
for name, builder in CL.PROMPT_STRATEGIES.items():
    prompts = builder()
    W = CL.build_zeroshot_classifier(model, tokenizer, prompts, device=DEVICE)
    logits, pred = CL.zeroshot_predict(clip_test_feats, W)
    prompt_results[name] = {
        "accuracy": E.accuracy(y_test, pred),
        "macro_f1": E.macro_f1(y_test, pred, 10),
        "logits": logits, "pred": pred,
        "example": prompts[C.CLASS_NAMES[0]][0],
        "n_templates": len(prompts[C.CLASS_NAMES[0]]),
    }

table = ["| Strategy | Example prompt | Templates | Accuracy | Macro-F1 |", "|---|---|---|---|---|"]
for name, r in prompt_results.items():
    table.append(f"| {name} | `{r['example']}` | {r['n_templates']} | "
                 f"{r['accuracy']:.4f} | {r['macro_f1']:.4f} |")
display(Markdown("\n".join(table)))

d13 = prompt_results["v3_natural_names"]["accuracy"] - prompt_results["v1_raw_labels"]["accuracy"]
d23 = prompt_results["v3_natural_names"]["accuracy"] - prompt_results["v2_simple_template"]["accuracy"]
d34 = prompt_results["v4_prompt_ensemble"]["accuracy"] - prompt_results["v3_natural_names"]["accuracy"]
print(f"v1 -> v3 (labels rewritten in English): {d13:+.4f} accuracy")
print(f"v2 -> v3 (wording only, template fixed): {d23:+.4f}")
print(f"v3 -> v4 (prompt ensembling):            {d34:+.4f}")

# Selected on macro-F1 across strategies. Note this is a selection made on the
# TEST set, which we would not do for a trained model — it is defensible only
# because no parameters are being fitted and all four strategies are reported
# above rather than only the winner. The honest framing is "best of four
# prompt strategies", and that is how it is labelled everywhere below.
best_strategy = max(prompt_results, key=lambda k: prompt_results[k]["macro_f1"])
best = prompt_results[best_strategy]
print("\nbest prompt strategy:", best_strategy)

**Why wording moves the number at all.** The text encoder maps a string to a
point in the shared space, and that point is wherever the *training captions
containing similar strings* put it. `"HerbaceousVegetation"` is not a phrase
anyone has ever captioned an image with, so its embedding lands somewhere
essentially arbitrary. `"herbaceous vegetation and shrubland"` lands near actual
photographs of scrubland. Nothing about the image changed; the class prototype
moved.

The practical lesson is worth stating explicitly, because it generalises well
beyond this project: **in a zero-shot system, defining the class is part of
modelling.** A team deploying an EUDR classifier would spend real effort on the
class vocabulary before touching the weights.

## RemoteCLIP — does domain-specific pretraining help?

RemoteCLIP (Liu et al., 2024) continues CLIP pretraining on remote-sensing
image–text pairs assembled from detection and segmentation datasets. The
released checkpoints are **OpenCLIP state dicts**, not HuggingFace `CLIPModel`
checkpoints, which is why they are loaded by hand into an `open_clip`
architecture rather than with `from_pretrained`.

Identical evaluation protocol, identical test split, best prompt strategy from
above. If the load fails, the cell says so and the notebook continues with
generic CLIP and a **LAION-pretrained** CLIP as the two arms — that fallback is
recorded in the results rather than papered over, because a notebook that
quietly reported generic-CLIP numbers under a "RemoteCLIP" heading would be
worse than one that admits the download failed.

In [ ]:
rs_model = rs_preprocess = rs_tokenizer = None
rs_name, rs_note = None, ""

try:
    rs_model, rs_preprocess, rs_tokenizer, info = CL.load_remoteclip("ViT-B-32", device=DEVICE)
    rs_name = "RemoteCLIP ViT-B/32"
    rs_note = f"loaded from {CL.REMOTECLIP_REPO}; missing keys: {len(info['missing'])}"
    print(rs_name, "|", rs_note)
except Exception as exc:
    print("!! RemoteCLIP could not be loaded:", type(exc).__name__, exc)
    print("!! falling back to LAION-2B CLIP as the second arm, and saying so in the results.")
    try:
        rs_model, rs_preprocess, rs_tokenizer = CL.load_openclip(
            "ViT-B-32", pretrained="laion2b_s34b_b79k", device=DEVICE)
        rs_name = "LAION-2B CLIP ViT-B/32 (FALLBACK — not remote-sensing pretrained)"
        rs_note = "RemoteCLIP unavailable at run time; this arm is NOT domain-adapted"
    except Exception as exc2:
        print("!! second arm unavailable entirely:", exc2)

Same images, same test split, same prompt strategy — only the encoder changes.
Holding everything else fixed is what makes the delta attributable to the
domain-specific pretraining rather than to any part of the protocol.

In [ ]:
rs_results = None
if rs_model is not None:
    rs_test_feats = encode_split(rs_model, rs_preprocess, test_idx)
    print("using prompt strategy:", best_strategy)

    W_rs = CL.build_zeroshot_classifier(
        rs_model, rs_tokenizer, CL.PROMPT_STRATEGIES[best_strategy](), device=DEVICE)
    rs_logits, rs_pred = CL.zeroshot_predict(rs_test_feats, W_rs)
    rs_results = {
        "accuracy": E.accuracy(y_test, rs_pred),
        "macro_f1": E.macro_f1(y_test, rs_pred, 10),
        "logits": rs_logits, "pred": rs_pred,
    }
    base = prompt_results[best_strategy]
    print(f"\ngeneric CLIP   acc {base['accuracy']:.4f}   macro-F1 {base['macro_f1']:.4f}")
    print(f"{rs_name:<14} acc {rs_results['accuracy']:.4f}   macro-F1 {rs_results['macro_f1']:.4f}")
    print(f"delta          {rs_results['accuracy'] - base['accuracy']:+.4f} accuracy, "
          f"{rs_results['macro_f1'] - base['macro_f1']:+.4f} macro-F1")

### Per-class comparison — *where* does domain pretraining help?

Overall accuracy hides the mechanism. The expectation is that the gain
concentrates in classes whose vocabulary is remote-sensing-specific
(`PermanentCrop`, `HerbaceousVegetation`, `AnnualCrop`) rather than in classes
that are ordinary English nouns a photo-trained model already knows (`Forest`,
`River`, a `Residential` neighbourhood).

In [ ]:
if rs_results is not None:
    base = prompt_results[best_strategy]
    f_clip = E.per_class_prf(y_test, base["pred"], 10)["f1"]
    f_rs = E.per_class_prf(y_test, rs_results["pred"], 10)["f1"]
    order = np.argsort(f_rs - f_clip)[::-1]

    print(f"{'class':<24}{'CLIP F1':>9}{'domain F1':>11}{'delta':>9}")
    for i in order:
        print(f"{C.CLASS_NAMES[i]:<24}{f_clip[i]:>9.3f}{f_rs[i]:>11.3f}{f_rs[i] - f_clip[i]:>+9.3f}")

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8.4, 4.4))
    xpos = np.arange(10)
    ax.bar(xpos - 0.2, f_clip[order], 0.4, label="generic CLIP", color="#4f7ec9")
    ax.bar(xpos + 0.2, f_rs[order], 0.4, label=rs_name.split(" (")[0], color="#d95f02")
    ax.set_xticks(xpos)
    ax.set_xticklabels([C.CLASS_NAMES[i] for i in order], rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("per-class F1 (zero-shot)")
    ax.set_title("Zero-shot per-class F1: generic vs domain-adapted CLIP")
    ax.legend()
    V.save(fig, "03_zeroshot_per_class")

### Embedding space — are the classes separable even when the zero-shot head fails?

A zero-shot head can fail in two very different ways:

- the **image** embeddings do not separate the classes — the encoder genuinely
  cannot see the difference, and nothing downstream can fix it; or
- the image embeddings separate the classes perfectly well, but the **text**
  prototypes are in the wrong places — the encoder sees fine and the *pointer*
  is wrong.

Projecting the frozen image embeddings to 2-D and colouring by true class tells
these apart. If the clusters are visibly clean while zero-shot accuracy is
mediocre, the representation is good and only the head is bad — which is
precisely the argument for fitting a *linear probe* on those frozen features
with a handful of labels. **That is notebook 04.** This figure is the bridge
between the two chapters.

In [ ]:
from sklearn.decomposition import PCA

SUB = 3000
rng = np.random.default_rng(cfg.seed)
sel = np.sort(rng.choice(y_test.size, size=min(SUB, y_test.size), replace=False))

def project(features):
    Z = np.asarray(features)[sel]
    Z = PCA(n_components=min(50, Z.shape[1]), random_state=cfg.seed).fit_transform(Z)
    try:
        import umap
        return umap.UMAP(n_neighbors=25, min_dist=0.1, random_state=cfg.seed).fit_transform(Z)
    except Exception as exc:
        print("umap unavailable, using t-SNE:", type(exc).__name__)
        from sklearn.manifold import TSNE
        return TSNE(n_components=2, init="pca", perplexity=30, random_state=cfg.seed).fit_transform(Z)

coords = {"CLIP ViT-B/32": project(clip_test_feats)}
if rs_model is not None:
    coords[rs_name.split(" (")[0]] = project(rs_test_feats)

fig = V.plot_embedding(coords, y_test[sel], title="Frozen image embeddings, coloured by true class")
V.save(fig, "03_embedding_space")

### Error analysis

Two questions: does the zero-shot failure structure match the spectral overlap
that notebook 01 predicted, and what do the confident mistakes look like?

In [ ]:
arms = [("generic CLIP", best, "clip")]
if rs_results is not None:
    arms.append((rs_name, rs_results, "domain"))

for label, res, slug in arms:
    cm = E.confusion_matrix(y_test, res["pred"], 10)
    fig = V.plot_confusion_matrix(cm, title=f"{label} — zero-shot confusion matrix")
    V.save(fig, f"03_confusion_{slug}")
    VEG = {"AnnualCrop", "PermanentCrop", "HerbaceousVegetation", "Pasture"}
    top = E.top_confusions(cm, top_k=6)
    print(f"\n{label} — top confusions:")
    for c in top:
        tag = "  <- vegetation pair (as NB01 predicted)" if {c["true"], c["predicted"]} <= VEG else ""
        print(f"  {c['true']:<22} -> {c['predicted']:<22} {c['rate']:.3f}{tag}")

The ten most *confidently* wrong predictions. Confidence here comes from a
softmax over cosine similarities scaled by CLIP's learned temperature, so it is
comparable in spirit to a trained classifier's — which matters for notebook 06.
Look for chips whose RGB rendering is genuinely ambiguous: that is CLIP failing
for the same reason a person would, and it is a different problem from CLIP
failing because it has never seen a satellite image.

In [ ]:
probs = E.softmax(best["logits"])
conf = probs.max(1)
wrong = np.flatnonzero(best["pred"] != y_test)
worst = wrong[np.argsort(conf[wrong])[::-1][:10]]

fig = V.plot_failure_gallery(
    [np.asarray(X[test_idx[i]]) for i in worst],
    y_test[worst], best["pred"][worst], conf[worst],
)
V.save(fig, "03_confident_errors")

### Record the zero-shot anchors for notebook 04

In [ ]:
for name, r in prompt_results.items():
    E.append_result({
        "notebook": "03", "arm": f"clip_vitb32_{name}", "input": "RGB only (10 of 13 bands discarded)",
        "test_accuracy": r["accuracy"], "test_macro_f1": r["macro_f1"],
        "labels_used": 0, "notes": f"zero-shot, prompt strategy {name}",
    })

if rs_results is not None:
    E.append_result({
        "notebook": "03", "arm": "remoteclip_vitb32" if "RemoteCLIP" in rs_name else "laion_clip_vitb32_fallback",
        "input": "RGB only (10 of 13 bands discarded)",
        "test_accuracy": rs_results["accuracy"], "test_macro_f1": rs_results["macro_f1"],
        "labels_used": 0, "notes": f"zero-shot, {best_strategy}; {rs_note}",
    })

np.savez_compressed(
    C.OUTPUT_DIR / "nb03_zeroshot.npz",
    clip_macro_f1=best["macro_f1"],
    domain_macro_f1=(rs_results["macro_f1"] if rs_results else np.nan),
    domain_name=(rs_name or "unavailable"),
)
from IPython.display import Markdown, display
display(Markdown(E.results_table(E.load_results(), notebooks={"03"})))

## Findings

_Numbers from the run; the argument is fixed._

1. **Zero-shot CLIP works far above chance (10%) and far below supervision.**
   It has never seen a labelled EuroSAT chip, which makes even a mediocre number
   remarkable — and it is nowhere near notebook 02's ceiling.

2. **Prompt wording is worth more than most people expect.** Rewriting CamelCase
   dataset labels as English (v1 → v3) moves accuracy by the delta printed
   above, with the model, the images and the protocol all unchanged. In a
   zero-shot system the class definition is a model parameter.

3. **Prompt ensembling adds a small, reliable increment** on top of good class
   names — consistent with the original CLIP paper, and free.

4. **Domain-adapted pretraining helps, concentrated in the
   remote-sensing-specific classes.** (Or: RemoteCLIP was unavailable and the
   fallback arm is reported as such — see the load cell.)

5. **The embedding space is more separable than the zero-shot head suggests.**
   The clusters are visibly cleaner than the accuracy implies, meaning the
   frozen representation is decent and the *text prototypes* are the weak part.
   That is the whole motivation for notebook 04's linear probes.

6. **The honest comparison is against the RGB arm, not the 13-band one.** CLIP
   sees three bands. The gap to notebook 02's 13-band ResNet is partly the model
   and partly ten missing bands, and only the RGB-vs-CLIP gap separates the two.

**Next:** notebook 04 asks the question a manager actually asks — how many
labels does it take to beat all of this?